# 00_context: What Is a Linguistic Specification?

**Lab report:** Invariant Specification Demystifies Translation

GLOSSOPETRAE develops languages, writing systems, speech, translation systems, and descendant language families from explicit specifications.

This notebook investigates the core object:

> specification

Questions:

- What information does a linguistic specification contain?
- What transformations preserve it?
- What changes when languages evolve?
- What remains invariant across speech, text, glyphs, translation, and descendants?

## 1. Clone and setup

This notebook treats the upstream GLOSSOPETRAE repository as the engine and this repo as the specification-analysis layer.

In [ ]:
from pathlib import Path
import os
import re
import json
import subprocess
import textwrap

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

print("Repository exists:", UPSTREAM_DIR.exists())

## 2. Repository map

Start by inspecting the upstream project structure. The goal is not to make claims yet, but to locate where language, script, speech, translation, and lineage are represented.

In [ ]:
for p in sorted(UPSTREAM_DIR.iterdir()):
    print(p.name)

## 3. Locate core linguistic objects

Search filenames and paths for likely concepts. This creates a first map of where the engine exposes linguistic structure.

In [ ]:
keywords = [
    "language",
    "translation",
    "phonology",
    "grammar",
    "script",
    "spec",
    "sound",
    "glyph",
    "daughter",
    "descendant",
    "evolution",
    "lineage",
]

matches = []
for root, dirs, files in os.walk(UPSTREAM_DIR):
    # Skip noisy directories
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__", ".venv", "node_modules"}]
    for f in files:
        path = Path(root) / f
        rel = path.relative_to(UPSTREAM_DIR)
        lower = str(rel).lower()
        hit = [k for k in keywords if k in lower]
        if hit:
            matches.append({"path": str(rel), "keywords": ", ".join(hit)})

matches[:20], len(matches)

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(matches).sort_values(["keywords", "path"])
    display(df.head(50))
except ImportError:
    for row in matches[:50]:
        print(row)

## 4. Search file contents for specification language

Now inspect text-bearing source and documentation files for explicit uses of `specification`, `language`, `translation`, and related terms.

In [ ]:
TEXT_SUFFIXES = {".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml", ".js", ".ts", ".html", ".css"}

content_hits = []
for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue
    lower = text.lower()
    for k in keywords:
        count = lower.count(k)
        if count:
            content_hits.append({
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "keyword": k,
                "count": count
            })

try:
    import pandas as pd
    hits_df = pd.DataFrame(content_hits)
    summary = hits_df.groupby(["keyword", "path"], as_index=False)["count"].sum()
    display(summary.sort_values(["keyword", "count"], ascending=[True, False]).head(80))
except Exception:
    for row in content_hits[:80]:
        print(row)

## 5. Preliminary concept table

Fill this table after inspecting the repository map. The goal is to separate observable outputs from the latent or explicit specification that generates them.

In [ ]:
concept_rows = [
    {
        "concept": "specification",
        "observable": "research question",
        "expected evidence": "configuration files, schemas, prompts, classes, or docs describing constraints",
    },
    {
        "concept": "language",
        "observable": "yes",
        "expected evidence": "generated word lists, grammar, lexicon, examples",
    },
    {
        "concept": "phonology / speech",
        "observable": "yes",
        "expected evidence": "phoneme inventory, sound rules, audio synthesis",
    },
    {
        "concept": "writing system / glyphs",
        "observable": "yes",
        "expected evidence": "script generator, SVG glyphs, orthography rules",
    },
    {
        "concept": "translation",
        "observable": "yes",
        "expected evidence": "mapping between meanings and language realizations",
    },
    {
        "concept": "descendant languages",
        "observable": "yes",
        "expected evidence": "sound change, lexical change, lineage, daughter languages",
    },
]

try:
    import pandas as pd
    display(pd.DataFrame(concept_rows))
except Exception:
    for row in concept_rows:
        print(row)

## 6. First generated example

Run the smallest upstream example that produces a language. Record:

1. the input specification,
2. generated language artifacts,
3. generated script or glyph artifacts,
4. translation artifacts,
5. descendant or evolved language artifacts.

This section is intentionally left as a work area because the exact command depends on the upstream repo interface.

In [ ]:
# TODO: replace with the smallest upstream command after inspecting README/docs.
# Example shape:
#
# subprocess.run(["python", "..."], cwd=UPSTREAM_DIR, check=True)
#
# Then inspect generated outputs.

## 7. Preliminary hypothesis

**Hypothesis 0**

A linguistic specification contains constraints that generate multiple representations of a language:

- text,
- speech,
- glyphs,
- translations,
- descendant languages.

The specification appears more fundamental than any individual representation.

The next notebooks should test whether this is actually true in the upstream engine.

## 8. Specification map

Save a simple diagram for the lab report.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

figures = ROOT / "figures"
figures.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis("off")

nodes = {
    "Specification\n(constraints)": (0.5, 0.85),
    "Language\n(structure + content)": (0.5, 0.65),
    "Text\n(orthography)": (0.2, 0.42),
    "Speech\n(phonology)": (0.5, 0.42),
    "Glyphs\n(writing system)": (0.8, 0.42),
    "Translations\n(other languages)": (0.35, 0.20),
    "Descendants\n(evolution)": (0.65, 0.20),
}

for label, (x, y) in nodes.items():
    ax.text(
        x, y, label,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.4", linewidth=1),
        fontsize=10
    )

edges = [
    ("Specification\n(constraints)", "Language\n(structure + content)"),
    ("Language\n(structure + content)", "Text\n(orthography)"),
    ("Language\n(structure + content)", "Speech\n(phonology)"),
    ("Language\n(structure + content)", "Glyphs\n(writing system)"),
    ("Text\n(orthography)", "Translations\n(other languages)"),
    ("Speech\n(phonology)", "Translations\n(other languages)"),
    ("Glyphs\n(writing system)", "Translations\n(other languages)"),
    ("Translations\n(other languages)", "Descendants\n(evolution)"),
    ("Language\n(structure + content)", "Descendants\n(evolution)"),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

path = figures / "00_specification_map.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 9. Questions for later notebooks

| Notebook | Question |
|---|---|
| 07_specifications | What exactly is specified? What is optional? What is required? |
| 13_translation | What survives translation? What changes? |
| 17_writing_systems | How are scripts generated? What is preserved across scripts? |
| 23_descendants | How are daughter languages derived from the same specification? |
| 29_invariants | Which properties remain invariant across all transformations? |